# S-CoT Distilled Model — Results & Analysis

This notebook downloads inference results generated on the TPU and displays them.

> **Why not run inference here?** The training used [Tunix](https://github.com/google/tunix),
> which is a **TPU-only** framework. The LoRA checkpoints are in JAX/Orbax format and
> require TPU hardware. Inference runs automatically on the TPU after training and the
> results are synced to GCS.

**Two models trained:**
- `sft-scot`: S-CoT structured reasoning (loss=1.27, ppl=3.55)
- `sft-flat`: Flat baseline (loss=0.275, ppl=1.32)

In [ ]:
# Cell 1: Authenticate and download results from GCS
from google.colab import auth
auth.authenticate_user()

!mkdir -p /content/scot_results
!gsutil -m cp -r gs://tpu-builder1-scot-checkpoints/ /content/scot_results/ 2>/dev/null || echo 'Bucket download failed'

import os
for root, dirs, files in os.walk('/content/scot_results'):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f'  {path} ({size:,} bytes)')

In [ ]:
# Cell 2: Display S-CoT inference results
import json
from IPython.display import HTML, display

def show_results(json_path, title):
    if not os.path.exists(json_path):
        print(f'File not found: {json_path}')
        print('Inference may not have run yet. Re-run the watchdog and check back.')
        return

    with open(json_path) as f:
        results = json.load(f)

    display(HTML(f'<h2>{title}</h2>'))
    for r in results:
        display(HTML(f'''
        <div style="border:1px solid #444; border-radius:8px; padding:16px; margin:12px 0; background:#1a1a2e;">
            <div style="color:#e94560; font-weight:bold; font-size:14px;">Q: {r["question"]}</div>
            <div style="color:#eee; margin-top:8px; white-space:pre-wrap; font-family:monospace; font-size:13px;">{r["answer"]}</div>
            <div style="color:#888; margin-top:8px; font-size:11px;">{r["num_tokens"]} tokens | {r["time_seconds"]}s</div>
        </div>
        '''))

# Try multiple possible paths
scot_paths = [
    '/content/scot_results/inference_scot.json',
    '/content/scot_results/tpu-builder1-scot-checkpoints/inference_scot.json',
]
for p in scot_paths:
    if os.path.exists(p):
        show_results(p, 'S-CoT Model Responses')
        break
else:
    print('S-CoT inference results not found yet.')
    print('The inference script runs after training on the TPU.')
    print('Re-run the watchdog and the results will appear here.')

In [ ]:
# Cell 3: Display Flat baseline inference results
flat_paths = [
    '/content/scot_results/inference_flat.json',
    '/content/scot_results/tpu-builder1-scot-checkpoints/inference_flat.json',
]
for p in flat_paths:
    if os.path.exists(p):
        show_results(p, 'Flat Baseline Responses')
        break
else:
    print('Flat inference results not found yet.')

In [ ]:
# Cell 4: Side-by-side comparison (if both results exist)
import json, os
from IPython.display import HTML, display

def load_results(name):
    for prefix in ['/content/scot_results', '/content/scot_results/tpu-builder1-scot-checkpoints']:
        p = f'{prefix}/inference_{name}.json'
        if os.path.exists(p):
            with open(p) as f:
                return json.load(f)
    return None

scot = load_results('scot')
flat = load_results('flat')

if scot and flat:
    display(HTML('<h2>Side-by-Side Comparison: S-CoT vs Flat</h2>'))
    for s, f in zip(scot, flat):
        display(HTML(f'''
        <div style="border:1px solid #444; border-radius:8px; padding:16px; margin:12px 0; background:#0f3460;">
            <div style="color:#e94560; font-weight:bold; font-size:14px;">Q: {s["question"]}</div>
            <div style="display:flex; gap:16px; margin-top:12px;">
                <div style="flex:1; background:#16213e; padding:12px; border-radius:6px;">
                    <div style="color:#00d2ff; font-weight:bold; margin-bottom:6px;">S-CoT ({s["num_tokens"]} tok)</div>
                    <div style="color:#eee; white-space:pre-wrap; font-family:monospace; font-size:12px;">{s["answer"][:500]}</div>
                </div>
                <div style="flex:1; background:#16213e; padding:12px; border-radius:6px;">
                    <div style="color:#ffa500; font-weight:bold; margin-bottom:6px;">Flat ({f["num_tokens"]} tok)</div>
                    <div style="color:#eee; white-space:pre-wrap; font-family:monospace; font-size:12px;">{f["answer"][:500]}</div>
                </div>
            </div>
        </div>
        '''))
else:
    print('Need both S-CoT and Flat results for comparison.')
    print('Run the watchdog to completion and re-download.')

In [ ]:
# Cell 5: Training metrics summary
from IPython.display import HTML, display

display(HTML('''
<h2>Training Summary</h2>
<table style="border-collapse:collapse; margin:12px 0; font-family:monospace;">
  <tr style="background:#16213e;">
    <th style="padding:8px 16px; border:1px solid #444; color:#e94560;">Metric</th>
    <th style="padding:8px 16px; border:1px solid #444; color:#00d2ff;">S-CoT Model</th>
    <th style="padding:8px 16px; border:1px solid #444; color:#ffa500;">Flat Baseline</th>
  </tr>
  <tr><td style="padding:8px 16px; border:1px solid #444;">Base Model</td><td style="padding:8px 16px; border:1px solid #444;">Qwen2.5-3B-Instruct</td><td style="padding:8px 16px; border:1px solid #444;">Qwen2.5-3B-Instruct</td></tr>
  <tr><td style="padding:8px 16px; border:1px solid #444;">LoRA Rank</td><td style="padding:8px 16px; border:1px solid #444;">16</td><td style="padding:8px 16px; border:1px solid #444;">16</td></tr>
  <tr><td style="padding:8px 16px; border:1px solid #444;">LoRA Alpha</td><td style="padding:8px 16px; border:1px solid #444;">32</td><td style="padding:8px 16px; border:1px solid #444;">32</td></tr>
  <tr><td style="padding:8px 16px; border:1px solid #444;">Training Steps</td><td style="padding:8px 16px; border:1px solid #444;">500</td><td style="padding:8px 16px; border:1px solid #444;">500</td></tr>
  <tr><td style="padding:8px 16px; border:1px solid #444;">Dataset Size</td><td style="padding:8px 16px; border:1px solid #444;">3,817</td><td style="padding:8px 16px; border:1px solid #444;">3,681</td></tr>
  <tr style="background:#1a1a2e;"><td style="padding:8px 16px; border:1px solid #444; font-weight:bold;">Final Loss</td><td style="padding:8px 16px; border:1px solid #444; color:#00d2ff; font-weight:bold;">1.27</td><td style="padding:8px 16px; border:1px solid #444; color:#ffa500; font-weight:bold;">0.275</td></tr>
  <tr style="background:#1a1a2e;"><td style="padding:8px 16px; border:1px solid #444; font-weight:bold;">Perplexity</td><td style="padding:8px 16px; border:1px solid #444; color:#00d2ff; font-weight:bold;">3.55</td><td style="padding:8px 16px; border:1px solid #444; color:#ffa500; font-weight:bold;">1.32</td></tr>
  <tr><td style="padding:8px 16px; border:1px solid #444;">Hardware</td><td style="padding:8px 16px; border:1px solid #444;">TPU v6e-8</td><td style="padding:8px 16px; border:1px solid #444;">TPU v6e-8</td></tr>
  <tr><td style="padding:8px 16px; border:1px solid #444;">Training Time</td><td style="padding:8px 16px; border:1px solid #444;">~2m 51s</td><td style="padding:8px 16px; border:1px solid #444;">~2m 51s</td></tr>
</table>
'''))